# DeBERTa-v3-FT Hard-Negative Augmentation — 5-Seed A100 Run

**Purpose:** Replicate the 5-seed DeBERTa-v3-FT + hard-negative augmentation experiment for Table 11.

**Workflow (run cells in order):**
1. **Cell 1** — verify A100 GPU
2. **Cell 2** — install pinned dependencies
3. **Cell 3** — mount Google Drive
4. **Cell 4** — extract dataset + source code
5. **Cell 5** — set working directory
6. **Cell 6** — verify hard-negative SHA-256 hashes
7. **Cell 7** — **seed-42 sanity check** (retrain baseline, compare vs recorded values)
8. **Cell 8** — **run 5 hard-negative augmented seeds** (only if Cell 7 exits 0)
9. **Cell 9** — print Table 11
10. **Cell 10** — save outputs to Google Drive

**Runtime:** A100 GPU.  Estimated time: ~20 min (sanity check) + ~5 × 25 min (augmented seeds) ≈ 2.5 h total.

**After completion:** Download `pids_results/multi_seed_runs/` from Drive to local repo.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\nWARNING: No GPU detected!")
    print("Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")

In [ ]:
# ── Cell 2: Install pinned dependencies ──────────────────────────────────
# Pinned to match the environment used for the original DeBERTa-v3 baseline
# runs (Colab CUDA, seeds 42/123/2024). Do not use -U; pinning is required
# for reproducibility before the seed-42 sanity check (Cell 7).
!pip install -q \
  transformers==4.40.0 \
  datasets==2.19.0 \
  scikit-learn==1.3.2 \
  evaluate==0.4.1 \
  accelerate==0.29.3
print("Dependencies installed (pinned).")
import transformers, datasets, sklearn, accelerate
print(f"  transformers : {transformers.__version__}")
print(f"  datasets     : {datasets.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  accelerate   : {accelerate.__version__}")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
# ── Cell 4: Extract dataset and source code ───────────────────────────────
import zipfile, os

PROJECT_DIR = '/content/pids_project'
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Extracting dataset...")
with zipfile.ZipFile('/content/drive/MyDrive/pids_bench_v3.zip', 'r') as z:
    z.extractall(PROJECT_DIR)
print("  Dataset extracted.")

print("Extracting source code...")
with zipfile.ZipFile('/content/drive/MyDrive/project_src.zip', 'r') as z:
    z.extractall(PROJECT_DIR)
print("  Source code extracted.")

checks = [
    f'{PROJECT_DIR}/data/pids_bench_v3/train.csv',
    f'{PROJECT_DIR}/data/pids_bench_v3/val.csv',
    f'{PROJECT_DIR}/data/pids_bench_v3/test.csv',
    f'{PROJECT_DIR}/data/pids_bench_v3/eval_subsets/hard_benign_test.csv',
    f'{PROJECT_DIR}/data/pids_bench_v3/eval_subsets/obfuscated_attacks.csv',
    f'{PROJECT_DIR}/data/pids_bench_v3/ood/domain_ood.csv',
    f'{PROJECT_DIR}/data/pids_bench_v3/ood/structural_ood.csv',
    # Hard-negative augmentation files
    f'{PROJECT_DIR}/data/pids_bench_v3/hard_negative_train.csv',
    f'{PROJECT_DIR}/data/pids_bench_v3/hard_negative_val.csv',
    f'{PROJECT_DIR}/src/baselines/deberta_v3.py',
    f'{PROJECT_DIR}/src/baselines/deberta_v3_hardneg.py',
    f'{PROJECT_DIR}/scripts/train_multi_seed.py',
    f'{PROJECT_DIR}/scripts/verify_seed42_sanity.py',
    f'{PROJECT_DIR}/scripts/analyze_hardneg_table11.py',
]
all_ok = True
for f in checks:
    exists = os.path.exists(f)
    print(f"  {'OK' if exists else 'MISSING'}  {f.replace(PROJECT_DIR, '.')}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll files present. Ready to proceed.")
else:
    print("\nERROR: Some files are missing. Check your zip files.")

In [ ]:
# ── Cell 5: Set working directory ────────────────────────────────────────
import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"Working directory : {os.getcwd()}")

In [ ]:
# ── Cell 6: Verify hard-negative SHA-256 hashes ───────────────────────────
import hashlib, os

EXPECTED = {
    'data/pids_bench_v3/hard_negative_train.csv':
        'bfdaaf940ba98f48ffea290ec345cd79e403fef9e28d96709d666ff92600216a',
    'data/pids_bench_v3/hard_negative_val.csv':
        '9f5f3351b864149b0cb61c2609a33b81604389956983caef88bb46e072e71137',
}

all_ok = True
for rel, expected in EXPECTED.items():
    path = os.path.join(os.getcwd(), rel)
    if not os.path.exists(path):
        print(f'  MISSING  {rel}')
        all_ok = False
        continue
    h = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    ok = (h == expected)
    print(f"  {'OK' if ok else 'MISMATCH'}  {rel}")
    if not ok:
        print(f"    expected : {expected}")
        print(f"    got      : {h}")
        all_ok = False

if all_ok:
    print('\nHash verification passed. Hard-negative files are intact.')
else:
    raise RuntimeError('Hash verification FAILED. Do not proceed.')

In [ ]:
# ── Cell 7: Seed-42 sanity check ──────────────────────────────────────────
# Retrains the DeBERTa-v3-FT baseline at seed 42 and compares against the
# recorded values in all_runs.csv.  This verifies environment reproducibility
# before running the 5 hard-negative augmented seeds.
#
# Expected runtime on A100: ~20 minutes.
# Exit 0 = proceed. Exit 1 = version divergence; investigate before continuing.
#
# NOTE: also saves pip freeze to:
#   outputs/multi_seed_runs/deberta_hardneg/pip_freeze_a100.txt
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'scripts/verify_seed42_sanity.py'],
    check=False,
)
if result.returncode != 0:
    raise RuntimeError(
        'Sanity check FAILED. See output above. '
        'Do NOT run Cell 8 until the environment divergence is resolved.'
    )
print('\nSanity check PASSED. Proceed to Cell 8.')

In [ ]:
# ── Cell 8: Run 5 DeBERTa-v3-FT hard-negative augmented seeds ────────────
# Run only after Cell 7 (sanity check) exits 0.
#
# Expected runtime on A100: ~25 min per seed × 5 seeds ≈ 2 hours.
# Seeds already recorded in all_runs.csv are skipped automatically.
# Outputs go to outputs/multi_seed_runs/deberta_hardneg/seed_{seed}/
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'scripts/train_multi_seed.py', '--models', 'deberta_hardneg'],
    check=False,
)
if result.returncode != 0:
    print('\nWARNING: train_multi_seed.py exited with non-zero code.')
    print('Check the output above. Partial results may still be usable.')
else:
    print('\nAll 5 seeds completed successfully.')

In [ ]:
# ── Cell 9: Print Table 11 ────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, 'scripts/analyze_hardneg_table11.py', '--partial-ok'],
    check=True,
)

In [ ]:
# ── Cell 10: Save outputs to Google Drive ─────────────────────────────────
import shutil, os

DRIVE_OUTPUT = '/content/drive/MyDrive/pids_results'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Save the full multi_seed_runs directory (includes all_runs.csv,
# per-seed summary.json / per_split_metrics.csv, and pip_freeze_a100.txt)
src = 'outputs/multi_seed_runs'
dst = f'{DRIVE_OUTPUT}/multi_seed_runs'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst, ignore=shutil.ignore_patterns('sanity_check_seed42/model', 'deberta_hardneg/*/model'))
print(f'Saved outputs/multi_seed_runs → {dst}')

# Save model checkpoints for each hardneg seed (large; ~1.4 GB each)
import glob
model_dirs = sorted(glob.glob('outputs/multi_seed_runs/deberta_hardneg/seed_*/model'))
for md in model_dirs:
    seed_label = os.path.basename(os.path.dirname(md))
    dst_model = f'{DRIVE_OUTPUT}/deberta_hardneg_models/{seed_label}'
    if os.path.exists(dst_model):
        shutil.rmtree(dst_model)
    shutil.copytree(md, dst_model)
    print(f'  model → {dst_model}')

print('\nDone. Sync complete.')
print(f'Download from Drive: {DRIVE_OUTPUT}')